# My First AI RAG Design

In [1]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [13]:
import fitz  # correct

doc = fitz.open("Downloads/MLtest.pdf")

print(doc)
print(len(doc))

Document('Downloads/MLtest.pdf')
579


In [22]:
page = doc[50]
print(page)

page 50 of <Downloads/MLtest.pdf, doc# 6>


In [23]:
text = page.get_text()

print(text)

1.2
Examples of Machine Learning Applications
11
to optimize a function1. Let us say we want to build a machine that roasts
coﬀee.
The machine has many inputs that aﬀect the quality: various
settings of temperatures, times, coﬀee bean type, and so forth. We make
a number of experiments and for diﬀerent settings of these inputs, we
measure the quality of the coﬀee, for example, as consumer satisfaction.
To ﬁnd the optimal setting, we ﬁt a regression model linking these inputs
to coﬀee quality and choose new points to sample near the optimum of
the current model to look for a better conﬁguration. We sample these
points, check quality, and add these to the data and ﬁt a new model. This
is generally called response surface design.
1.2.4
Unsupervised Learning
In supervised learning, the aim is to learn a mapping from the input to
an output whose correct values are provided by a supervisor. In unsuper-
vised learning, there is no such supervisor and we only have input data.
The aim is to ﬁnd

In [29]:
import fitz

doc = fitz.open("Downloads/MLtest.pdf")

for page in doc:
    text = page.get_text()
    print(text)

Introduction
to
Machine
Learning
Second
Edition

Adaptive Computation and Machine Learning
Thomas Dietterich, Editor
Christopher Bishop, David Heckerman, Michael Jordan, and Michael
Kearns, Associate Editors
A complete list of books published in The Adaptive Computation and
Machine Learning series appears at the back of this book.

Introduction
to
Machine
Learning
Second
E d i t i o n
Ethem Alpaydın
The MIT Press
Cambridge, Massachusetts
London, England

© 2010 Massachusetts Institute of Technology
All rights reserved. No part of this book may be reproduced in any form by any
electronic or mechanical means (including photocopying, recording, or informa-
tion storage and retrieval) without permission in writing from the publisher.
For information about special quantity discounts, please email
special_sales@mitpress.mit.edu.
Typeset in 10/13 Lucida Bright by the author using LATEX 2ε.
Printed and bound in the United States of America.
Library of Congress Cataloging-in-Publication Informa

In [80]:
import fitz

doc = fitz.open("Downloads/MLtest.pdf")

full_text = ""

for page in doc:
    text = page.get_text()
    full_text += text + "\n"


len(full_text)
#print(full_text[:5000])

944922

In [59]:
import fitz

doc = fitz.open("Downloads/MLtest.pdf")

pages = []

for i, page in enumerate(doc):
    text = page.get_text()
    pages.append({
        "page": i + 1,
        "text": text
    })

print(pages[0])

{'page': 1, 'text': 'Introduction\nto\nMachine\nLearning\nSecond\nEdition\n'}


In [87]:
# ── Improved Text Cleaning ───────────────────────────────────
import re

def clean_text(text):
    # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def is_useful_page(text):
    """
    Returns False for pages we want to skip:
    - References pages
    - Table of contents pages
    - Index pages
    - Copyright pages
    - Very short pages
    """
    text_lower = text.lower()
    words      = text.split()

    # Skip very short pages
    if len(words) < 30:
        return False

    # Skip reference pages
    if text_lower.count("references") > 2:
        return False

    # Skip table of contents pages
    if text_lower.count("contents") > 2:
        return False

    # Skip index pages
    if text_lower.count("index") > 3:
        return False

    # Skip copyright pages
    if "all rights reserved" in text_lower:
        return False

    # Skip pages with too many numbers (likely index or TOC)
    number_count = len(re.findall(r'\b\d+\b', text))
    if number_count > 50:
        return False

    return True

# ── Apply improved cleaning ──────────────────────────────────
cleaned_pages = []
skipped       = 0

for page in pages:
    cleaned = clean_text(page["text"])

    if is_useful_page(cleaned):
        cleaned_pages.append({
            "page" : page["page"],
            "text" : cleaned
        })
    else:
        skipped += 1

print(f"✅ Pages after filtering : {len(cleaned_pages)}")
print(f"   Skipped pages        : {skipped}")
print(f"\nSample cleaned page:")
print(cleaned_pages[5]["text"][:500])

✅ Pages after filtering : 482
   Skipped pages        : 97

Sample cleaned page:
Figures 1.1 Example of a training dataset where each circle corresponds to one data instance with input values in the corresponding axes and its sign indicates the class. 6 1.2 A training dataset of used cars and the function ﬁtted. 10 2.1 Training set for the class of a “family car.” 22 2.2 Example of a hypothesis class. 23 2.3 C is the actual class and h is our induced hypothesis. 25 2.4 S is the most speciﬁc and G is the most general hypothesis. 26 2.5 We choose the hypothesis with the larges


In [89]:
# ── Step 8 (Improved): Better chunking strategy ─────────────
def chunk_text(cleaned_pages, chunk_size=300, overlap=50):
    chunks = []
    
    # Combine ALL pages into one big text first
    # then chunk across page boundaries
    all_words = []
    word_page_map = []  # tracks which page each word came from
    
    for page in pages:
        words = page["text"].split()
        all_words.extend(words)
        word_page_map.extend([page["page"]] * len(words))
    
    print(f"Total words in document: {len(all_words)}")
    
    # Slide through ALL words across all pages
    for i in range(0, len(all_words), chunk_size - overlap):
        chunk_words = all_words[i : i + chunk_size]
        chunk_text  = " ".join(chunk_words)
        
        # Get the page number of the first word in this chunk
        page_num = word_page_map[i] if i < len(word_page_map) else word_page_map[-1]
        
        if chunk_text.strip():
            chunks.append({
                "page"     : page_num,
                "chunk_id" : len(chunks) + 1,
                "text"     : chunk_text
            })
    
    return chunks

# Run improved chunking
# Re-run chunking on cleaned pages
chunks = chunk_text(cleaned_pages, chunk_size=300, overlap=50)

print(f"✅ Total chunks after cleaning: {len(chunks)}")
print(f"\nSample chunk 1:")
print(chunks[0])
print(f"\nSample chunk 2:")
print(chunks[1])
print(f"\nSample chunk 3:")
print(chunks[2])

Total words in document: 167740
✅ Total chunks after cleaning: 671

Sample chunk 1:
{'page': 1, 'chunk_id': 1, 'text': 'Introduction to Machine Learning Second Edition Adaptive Computation and Machine Learning Thomas Dietterich, Editor Christopher Bishop, David Heckerman, Michael Jordan, and Michael Kearns, Associate Editors A complete list of books published in The Adaptive Computation and Machine Learning series appears at the back of this book. Introduction to Machine Learning Second E d i t i o n Ethem Alpaydın The MIT Press Cambridge, Massachusetts London, England © 2010 Massachusetts Institute of Technology All rights reserved. No part of this book may be reproduced in any form by any electronic or mechanical means (including photocopying, recording, or informa- tion storage and retrieval) without permission in writing from the publisher. For information about special quantity discounts, please email special_sales@mitpress.mit.edu. Typeset in 10/13 Lucida Bright by the author usi

In [ ]:
# Install required libraries
pip install sentence-transformers
pip install faiss-cpu

In [90]:
# ============================================================
# STEP 9 — Create Embeddings
# ============================================================
from sentence_transformers import SentenceTransformer

# Load the embedding model
# all-MiniLM-L6-v2 is fast, lightweight and works great for RAG
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Embedding model loaded successfully")
print(f"   Model: all-MiniLM-L6-v2")

# Extract just the text from each chunk
chunk_texts = [chunk["text"] for chunk in chunks] 

print(f"\n⏳ Creating embeddings for {len(chunk_texts)} chunks...")
print("   This may take a few minutes...")

# Create embeddings for all chunks
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,    # shows progress as it runs
    batch_size=32              # process 32 chunks at a time
)

print(f"\n✅ Embeddings created successfully!")
print(f"   Total embeddings : {len(embeddings)}")
print(f"   Embedding shape  : {embeddings.shape}")
print(f"   Each embedding   : {embeddings.shape[1]} numbers")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded successfully
   Model: all-MiniLM-L6-v2

⏳ Creating embeddings for 671 chunks...
   This may take a few minutes...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]


✅ Embeddings created successfully!
   Total embeddings : 671
   Embedding shape  : (671, 384)
   Each embedding   : 384 numbers


In [96]:
# ============================================================
# STEP 10 — Store Embeddings in FAISS Vector Database
# ============================================================
import faiss
import numpy as np
from sklearn.preprocessing import normalize

# ── Convert embeddings to float32 first ─────────────────────
embeddings_np = np.array(embeddings).astype('float32')
print(f"Embeddings converted to float32: {embeddings_np.shape}")

# ── Normalize embeddings for cosine similarity ───────────────
embeddings_normalized = normalize(embeddings_np, norm='l2')

# ── Create FAISS index with Cosine Similarity ────────────────
dimension = embeddings_normalized.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_normalized)

print(f"✅ FAISS index rebuilt with Cosine Similarity")
print(f"   Total vectors stored : {index.ntotal}")

# ── Search Function ──────────────────────────────────────────
def search_chunks(question, k=3):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype('float32')

    # Normalize question embedding too
    question_embedding = normalize(question_embedding, norm='l2')

    # Search — higher score = more relevant
    scores, indices = index.search(question_embedding, k * 3)

    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        text  = chunk["text"]

        if len(text.split()) < 30:
            continue
        if text.count(".") > 15 and len(text.split()) < 100:
            continue

        results.append({
            "chunk_id" : chunk["chunk_id"],
            "page"     : chunk["page"],
            "score"    : scores[0][i],
            "text"     : chunk["text"]
        })

        if len(results) == k:
            break

    return results

# ── Test with cosine similarity ──────────────────────────────
question = "What is machine learning?"
results  = search_chunks(question, k=3)

print(f"Question: '{question}'")
print(f"\nTop {len(results)} most relevant chunks:\n")

for i, result in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Chunk ID : {result['chunk_id']}")
    print(f"Page     : {result['page']}")
    print(f"Score    : {result['score']:.4f}")
    print(f"Text     : {result['text'][:300]}...")
    print()

Embeddings converted to float32: (671, 384)
✅ FAISS index rebuilt with Cosine Similarity
   Total vectors stored : 671
Question: 'What is machine learning?'

Top 3 most relevant chunks:

--- Result 1 ---
Chunk ID : 31
Page     : 43
Score    : 0.6691
Text     : their past data to build models to use in credit applications, fraud detection, and the stock market. In manufacturing, learning models are used for optimiza- tion, control, and troubleshooting. In medicine, learning programs are used for medical diagnosis. In telecommunications, call patterns are a...

--- Result 2 ---
Chunk ID : 32
Page     : 43
Score    : 0.5852
Text     : image is not just a random collection of pixels; a face has structure. It is symmetric. There are the eyes, the nose, the mouth, located in certain places on the face. Each person’s face is a pattern composed of a particular combination of these. By analyzing sample face images of a person, a learni...

--- Result 3 ---
Chunk ID : 30
Page     : 42
Score    :

In [119]:
# ============================================================
# STEP 11 — Connect Groq LLM
# ============================================================
from groq import Groq
from dotenv import load_dotenv
import os
import re

# ── Load API key from environment ────────────────────────────
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# ── Set up Groq client ───────────────────────────────────────
client = Groq(api_key=api_key)
print("✅ Groq client connected successfully")

# ── Helper function to clean model response ──────────────────
def get_clean_response(response):
    content = response.choices[0].message.content
    # Remove the <think>...</think> block
    content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
    content = content.strip()
    return content

# ── Test with a simple question ──────────────────────────────
response = client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=[
        {"role": "user", "content": "who won the fifa world cup 2022 "}
    ]
)

print("✅ Test response received")
print(f"\nGroq says: {get_clean_response(response)}")  # ← use clean response here

✅ Groq client connected successfully
✅ Test response received

Groq says: **Argentina** won the 2022 FIFA World Cup. They defeated France 4–2 in a penalty shootout after the final ended 3–3 following extra time. The tournament was hosted in Qatar in December 2022.


In [141]:
# ============================================================
# STEP 12 — Full RAG Answer Function (Final Version)
# ============================================================
import re

def ask_rag(question, k=6):
    print(f"🔍 Searching for relevant chunks...")

    results = search_chunks(question, k=k)

    if not results:
        return "Sorry, I could not find any relevant information in the document."

    context = ""
    sources = []

    for i, result in enumerate(results):
        context += f"\n--- Context {i+1} (Page {result['page']}) ---\n"
        context += result["text"]
        context += "\n"
        sources.append(result["page"])

    print(f"✅ Found {len(results)} relevant chunks")
    print(f"   From pages: {sources}")

    prompt = f"""Answer the question below using ONLY the provided context.
Be concise — answer in 3 to 5 sentences maximum.
Always mention which page the information came from.
If the answer is not in the context, say "I could not find this information in the provided context."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER (3-5 sentences only):"""

    print(f"⏳ Generating answer...")

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",   # ✅ switched to non-thinking model
        messages=[
            {
                "role": "system",
                "content": "You are a concise assistant. Answer in 3-5 sentences only using the provided context. Just give the final answer directly with no extra explanation."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=1024
    )

    # ── Clean response ────────────────────────────────────────
    raw_answer = response.choices[0].message.content
    clean_answer = re.sub(r'<think>[\s\S]*?</think>', '', raw_answer)
    clean_answer = re.sub(r'</?think>', '', clean_answer)
    clean_answer = re.sub(r'\n{3,}', '\n\n', clean_answer)
    clean_answer = clean_answer.strip()

    print("\n" + "=" * 60)
    print("📚 QUESTION:")
    print(f"   {question}")
    print("\n📄 SOURCES (Pages used):")
    print(f"   {sources}")
    print("\n🤖 ANSWER:")
    print(f"   {clean_answer}")
    print("=" * 60)

    return clean_answer

In [142]:
answer = ask_rag("What is the difference between classification and regression?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [49, 61, 223, 50, 196, 48]
⏳ Generating answer...

📚 QUESTION:
   What is the difference between classification and regression?

📄 SOURCES (Pages used):
   [49, 61, 223, 50, 196, 48]

🤖 ANSWER:
   Classification and regression are both supervised learning problems where an input \(X\) is mapped to an output \(Y\) (Context 1, Page 49). In classification the output \(Y\) is a class code (e.g., 0/1), whereas in regression the output is a continuous number (Context 1, Page 49). The model \(g(x|\theta)\) is optimized to minimize approximation error in both cases, but the nature of the target variable differs (Context 1, Page 49). Thus, classification predicts discrete categories, while regression predicts real‑valued quantities.


In [143]:
# ── Test again ────────────────────────────────────────────────
answer = ask_rag("What is supervised learning?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [196, 54, 43, 43, 61, 49]
⏳ Generating answer...

📚 QUESTION:
   What is supervised learning?

📄 SOURCES (Pages used):
   [196, 54, 43, 43, 61, 49]

🤖 ANSWER:
   Supervised learning is a type of machine‑learning problem in which each training example consists of an input vector X and a corresponding output Y. The goal is to learn a mapping g(·) that predicts Y from X, minimizing the approximation error on the training data. This framework covers both regression (where Y is continuous) and classification (where Y is a class label). The learning algorithm optimizes model parameters θ so that the predicted values are as close as possible to the observed outputs (see page 49 and page 61).


In [144]:
answer = ask_rag('What Is Machine Learning?')

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [43, 43, 42, 56, 49, 55]
⏳ Generating answer...

📚 QUESTION:
   What Is Machine Learning?

📄 SOURCES (Pages used):
   [43, 43, 42, 56, 49, 55]

🤖 ANSWER:
   Machine learning is the practice of programming computers to optimize a performance criterion using example data or past experience (Page 43). It involves building mathematical models—often predictive or descriptive—whose parameters are tuned to fit training data (Page 43). The core task is to make inference from a sample, leveraging statistical theory to learn patterns that can be used for prediction or knowledge extraction (Page 42). In supervised learning, for instance, the goal is to learn a mapping from inputs \(X\) to outputs \(Y\) by minimizing approximation error (Page 49).


In [145]:
answer = ask_rag("What is a decision tree?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [225, 226, 344, 245, 246, 227]
⏳ Generating answer...

📚 QUESTION:
   What is a decision tree?

📄 SOURCES (Pages used):
   [225, 226, 344, 245, 246, 227]

🤖 ANSWER:
   A decision tree is a hierarchical, non‑parametric model for supervised learning that recursively partitions the input space using simple test functions at each internal node (Page 225). Each node applies a test fm(x) and routes the instance to one of its branches, eventually reaching a leaf that outputs a class label or numeric value (Page 226). The tree grows by selecting the best split at each step, often using impurity measures for classification, and stops when further splits would not improve purity (Page 227). Decision trees are popular because they are fast to learn and interpret, and they can handle both numeric and discrete features without conversion (Page 245).


In [146]:
answer = ask_rag("What is overfitting in machine learning?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [79, 298, 43, 49, 517, 48]
⏳ Generating answer...

📚 QUESTION:
   What is overfitting in machine learning?

📄 SOURCES (Pages used):
   [79, 298, 43, 49, 517, 48]

🤖 ANSWER:
   Overfitting in machine learning happens when the hypothesis class \(H\) is too complex for the available data, causing the model to learn not only the underlying function but also the noise in the training set. This results in a hypothesis that fits the training data very well but generalizes poorly to new examples. As the model complexity increases, the training error decreases, but beyond a certain point the generalization error starts to rise again (Page 79). Early stopping and cross‑validation are common techniques to prevent overfitting by limiting the effective complexity of the model (Page 298).


In [147]:
answer = ask_rag("What is cross‑validation ?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [80, 517, 123, 528, 461, 528]
⏳ Generating answer...

📚 QUESTION:
   What is cross‑validation ?

📄 SOURCES (Pages used):
   [80, 517, 123, 528, 461, 528]

🤖 ANSWER:
   Cross‑validation is a model‑selection technique that repeatedly partitions a dataset into training and validation sets to evaluate a model’s performance. For each split, the model is trained on the training set, its error is computed on the validation set, and the best model (e.g., the polynomial with the lowest validation error) is chosen (Page 80). This procedure ensures that the validation data are not reused for training, preventing bias in the error estimate (Page 517). Variants such as 5 × 2 cross‑validation or k‑fold cross‑validation are common implementations (Page 528).


In [148]:
answer = ask_rag("What is the k-nearest neighbour algorithm?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [212, 21, 213, 188, 197, 212]
⏳ Generating answer...

📚 QUESTION:
   What is the k-nearest neighbour algorithm?

📄 SOURCES (Pages used):
   [212, 21, 213, 188, 197, 212]

🤖 ANSWER:
   The k‑nearest neighbour (k‑NN) algorithm is a non‑parametric classification method that assigns a new instance to the class that appears most frequently among its \(k\) closest training examples. All neighbours contribute equally (unweighted vote) and ties are broken arbitrarily or by a weighted scheme. The distance metric (often Euclidean) determines the nearest neighbours, and \(k\) is usually chosen odd to minimise ties. The special case \(k=1\) reduces to a simple nearest‑neighbour classifier that partitions space into a Voronoi tessellation. This description is found on page 212.


In [153]:
answer = ask_rag("mention the kind of machine learning model discussion in this book?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [43, 32, 33, 56, 577, 49]
⏳ Generating answer...

📚 QUESTION:
   mention the kind of machine learning model discussion in this book?

📄 SOURCES (Pages used):
   [43, 32, 33, 56, 577, 49]

🤖 ANSWER:
   The book discusses supervised learning models, where a mapping \(y=g(x|\theta)\) is learned from input \(X\) to output \(Y\) (Page 49). It gives examples of linear regression \(y=wx+w_0\), quadratic \(y=w_2x^2+w_1x+w_0\), and higher‑order polynomial models (Page 49). It also covers kernel‑based models that use kernel functions to represent problems in a higher‑dimensional space (Page 33). Additionally, Bayesian methods with appropriately chosen priors are mentioned as part of the theoretical advances (Page 33).


# Building an App for the RAG system

In [154]:
# ============================================================
# STEP 13 — Save FAISS Index and Chunks to Disk
# ============================================================
import faiss
import pickle
import os

# ── Create a folder to save everything ──────────────────────
save_folder = "rag_index"
os.makedirs(save_folder, exist_ok=True)

# ── Save FAISS index ─────────────────────────────────────────
faiss.write_index(index, f"{save_folder}/faiss_index.bin")
print(f"✅ FAISS index saved to {save_folder}/faiss_index.bin")

# ── Save chunks (so we can retrieve text later) ──────────────
with open(f"{save_folder}/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)
print(f"✅ Chunks saved to {save_folder}/chunks.pkl")

# ── Save embedding model name ────────────────────────────────
with open(f"{save_folder}/config.pkl", "wb") as f:
    pickle.dump({
        "embedding_model" : "all-MiniLM-L6-v2",
        "total_chunks"    : len(chunks),
        "total_pages"     : len(pages)
    }, f)
print(f"✅ Config saved to {save_folder}/config.pkl")

print(f"\n📁 All files saved in '{save_folder}/' folder:")
for file in os.listdir(save_folder):
    size = os.path.getsize(f"{save_folder}/{file}") / 1024
    print(f"   {file} — {size:.1f} KB")

✅ FAISS index saved to rag_index/faiss_index.bin
✅ Chunks saved to rag_index/chunks.pkl
✅ Config saved to rag_index/config.pkl

📁 All files saved in 'rag_index/' folder:
   chunks.pkl — 1140.0 KB
   config.pkl — 0.1 KB
   faiss_index.bin — 1006.5 KB


In [155]:
# ============================================================
# STEP 14 — Load FAISS Index from Disk
# ============================================================
import faiss
import pickle
import os
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import numpy as np

save_folder = "rag_index"

# ── Check if saved files exist ───────────────────────────────
required_files = ["faiss_index.bin", "chunks.pkl", "config.pkl"]
all_exist = all(
    os.path.exists(f"{save_folder}/{f}") for f in required_files
)

if not all_exist:
    print("❌ Saved index not found — please run the previous step first")
else:
    # ── Load config first ────────────────────────────────────
    with open(f"{save_folder}/config.pkl", "rb") as f:
        config = pickle.load(f)

    print(f"✅ Config loaded:")
    print(f"   Embedding model : {config['embedding_model']}")
    print(f"   Total chunks    : {config['total_chunks']}")
    print(f"   Total pages     : {config['total_pages']}")

    # ── Load FAISS index ─────────────────────────────────────
    index = faiss.read_index(f"{save_folder}/faiss_index.bin")
    print(f"\n✅ FAISS index loaded:")
    print(f"   Total vectors   : {index.ntotal}")

    # ── Load chunks ──────────────────────────────────────────
    with open(f"{save_folder}/chunks.pkl", "rb") as f:
        chunks = pickle.load(f)
    print(f"\n✅ Chunks loaded:")
    print(f"   Total chunks    : {len(chunks)}")
    print(f"   Sample chunk    : {chunks[0]['text'][:100]}...")

    # ── Load embedding model ─────────────────────────────────
    embedding_model = SentenceTransformer(config['embedding_model'])
    print(f"\n✅ Embedding model loaded: {config['embedding_model']}")

    print("\n🎉 RAG system fully loaded and ready!")
    print("   You can now ask questions without rebuilding!")

✅ Config loaded:
   Embedding model : all-MiniLM-L6-v2
   Total chunks    : 671
   Total pages     : 579

✅ FAISS index loaded:
   Total vectors   : 671

✅ Chunks loaded:
   Total chunks    : 671
   Sample chunk    : Introduction to Machine Learning Second Edition Adaptive Computation and Machine Learning Thomas Die...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


✅ Embedding model loaded: all-MiniLM-L6-v2

🎉 RAG system fully loaded and ready!
   You can now ask questions without rebuilding!


In [159]:
# ============================================================
# STEP 15 — Add Conversation History
# ============================================================

# ── Initialize conversation history ──────────────────────────
conversation_history = []

def ask_rag_with_history(question, k=6):
    global conversation_history

    print(f"🔍 Searching for relevant chunks...")

    # ── Search FAISS for relevant chunks ─────────────────────
    results = search_chunks(question, k=k)

    if not results:
        return "Sorry, I could not find any relevant information in the document."

    # ── Build context from chunks ─────────────────────────────
    context = ""
    sources = []

    for i, result in enumerate(results):
        context += f"\n--- Context {i+1} (Page {result['page']}) ---\n"
        context += result["text"]
        context += "\n"
        sources.append(result["page"])

    print(f"✅ Found {len(results)} relevant chunks")
    print(f"   From pages: {sources}")

    # ── Build system message ──────────────────────────────────
    system_message = """You are a concise assistant that answers 
questions based strictly on the provided context from a Machine 
Learning textbook. Answer in 3-5 sentences only. Always mention 
which page the information came from. If the answer is not in 
the context say 'I could not find this information in the 
provided context.'"""

    # ── Build messages with history ───────────────────────────
    # Start with system message
    messages = [{"role": "system", "content": system_message}]

    # Add previous conversation history
    messages.extend(conversation_history)

    # Add current question with context
    current_prompt = f"""CONTEXT:
{context}

QUESTION: {question}

ANSWER (3-5 sentences only):"""

    messages.append({"role": "user", "content": current_prompt})

    print(f"⏳ Generating answer...")
    print(f"   History length: {len(conversation_history)} messages")

    # ── Send to Groq ──────────────────────────────────────────
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        temperature=0.1,
        max_tokens=512
    )

    # ── Clean response ────────────────────────────────────────
    raw_answer  = response.choices[0].message.content
    clean_answer = re.sub(r'<think>[\s\S]*?</think>', '', raw_answer)
    clean_answer = re.sub(r'</?think>', '', clean_answer)
    clean_answer = re.sub(r'\n{3,}', '\n\n', clean_answer)
    clean_answer = clean_answer.strip()

    # ── Save to conversation history ──────────────────────────
    conversation_history.append({
        "role"   : "user",
        "content": f"QUESTION: {question}"
    })
    conversation_history.append({
        "role"   : "assistant",
        "content": clean_answer
    })

    # ── Keep history to last 10 messages (5 exchanges) ────────
    # Prevents context window from getting too large
    if len(conversation_history) > 10:
        conversation_history = conversation_history[-10:]

    print("\n" + "=" * 60)
    print("📚 QUESTION:")
    print(f"   {question}")
    print("\n📄 SOURCES (Pages used):")
    print(f"   {sources}")
    print("\n🤖 ANSWER:")
    print(f"   {clean_answer}")
    print("=" * 60)

    return clean_answer

# ── Function to clear history ─────────────────────────────────
def clear_history():
    global conversation_history
    conversation_history = []
    print("✅ Conversation history cleared!")

In [160]:
# Question 1
answer1 = ask_rag_with_history("What is supervised learning?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [196, 54, 43, 43, 61, 49]
⏳ Generating answer...
   History length: 0 messages

📚 QUESTION:
   What is supervised learning?

📄 SOURCES (Pages used):
   [196, 54, 43, 43, 61, 49]

🤖 ANSWER:
   Supervised learning is a type of machine‑learning problem in which each training example consists of an input vector X and a corresponding output Y. The goal is to learn a mapping g(·) that predicts Y from X, minimizing the approximation error on the training data. This framework underlies both regression (where Y is continuous) and classification (where Y is a class label). The learning algorithm optimizes model parameters θ so that the predicted values are as close as possible to the true outputs. (Page 49)


In [161]:
# Question 2 — refers back to Question 1
answer2 = ask_rag_with_history("Give me an example of it")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [41, 480, 42, 54, 43, 82]
⏳ Generating answer...
   History length: 2 messages

📚 QUESTION:
   Give me an example of it

📄 SOURCES (Pages used):
   [41, 480, 42, 54, 43, 82]

🤖 ANSWER:
   An example of supervised learning is the spam‑email classifier described on page 41. In this task, each training example consists of an email document (the input) and a label indicating whether the message is spam or not (the output). The learning algorithm uses thousands of such labeled emails to learn a mapping that predicts the spam label for new, unseen messages. This illustrates how supervised learning extracts a predictive rule from labeled data. (Page 41)


In [162]:
# Question 3 — new topic
answer3 = ask_rag_with_history("What is a decision tree?")

🔍 Searching for relevant chunks...
✅ Found 6 relevant chunks
   From pages: [225, 226, 344, 245, 246, 227]
⏳ Generating answer...
   History length: 4 messages

📚 QUESTION:
   What is a decision tree?

📄 SOURCES (Pages used):
   [225, 226, 344, 245, 246, 227]

🤖 ANSWER:
   A decision tree is a hierarchical, non‑parametric model for supervised learning that partitions the input space into smaller regions through a sequence of recursive splits. Each internal node applies a simple test function \(f_m(x)\) on one or more input variables, directing the instance to one of its child branches; the leaves then output a class label (for classification) or a numeric value (for regression). The tree structure is learned greedily by selecting splits that maximize purity or reduce impurity, and the resulting model is interpretable as a set of IF‑THEN rules (Page 225‑226). Decision trees can handle both numeric and discrete features, require only the tree structure and node parameters to be stored, a

In [163]:
# Clear history when starting fresh
clear_history()

✅ Conversation history cleared!
